# Salmonella Metadata Preprocessing

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tungpnt-nt/AI-AMR_original/blob/main/Salmonella_Metadata_Preprocessing_Pipeline.ipynb)

This notebook prepares *Salmonella enterica* AMR metadata collected from **NCBI** and **BV-BRC**.

### Inputs
- `Salmonella_NCBI.csv`
- `Salmonella_BVBRC.csv`

### Required fields
`BioSample`, `Antibiotic`, `Resistant Phenotype`, `Species`, `Strain`, `Assembly`, `Isolation source`, `Isolation Country`


In [ ]:
import os
from pathlib import Path

import pandas as pd

OUTPUT_DIR = Path("data_processed")
OUTPUT_DIR.mkdir(exist_ok=True)

REQUIRED_COLUMNS = [
    "BioSample",
    "Antibiotic",
    "Resistant Phenotype",
    "Species",
    "Strain",
    "Assembly",
    "Isolation source",
    "Isolation Country",
]


## 1. Upload raw metadata

Upload the NCBI and BV-BRC metadata files. The notebook accepts `.csv` or `.xlsx` files.


In [ ]:
from google.colab import files

uploaded = files.upload()

ncbi_file = next(
    (name for name in uploaded if "NCBI" in name.upper()),
    None,
)
bvbrc_file = next(
    (name for name in uploaded if "BVBRC" in name.upper() or "BV-BRC" in name.upper()),
    None,
)

if ncbi_file is None or bvbrc_file is None:
    raise ValueError(
        "Please upload one NCBI file and one BV-BRC/BVBRC file."
    )

print("NCBI:", ncbi_file)
print("BV-BRC:", bvbrc_file)


## 2. Validate and standardize metadata

The two datasets are checked for the required columns. Antibiotic names and phenotype labels are standardized before combining the datasets.


In [ ]:
PHENOTYPE_MAP = {
    "S": "S",
    "SUSCEPTIBLE": "S",
    "SENSITIVE": "S",
    "I": "I",
    "INTERMEDIATE": "I",
    "R": "R",
    "RESISTANT": "R",
}


def load_table(path):
    suffix = Path(path).suffix.lower()
    if suffix == ".csv":
        return pd.read_csv(path, dtype="string")
    if suffix in {".xlsx", ".xls"}:
        return pd.read_excel(path, dtype="string")
    raise ValueError(f"Unsupported file format: {suffix}")


def clean_text(series):
    return (
        series.astype("string")
        .str.strip()
        .replace("", pd.NA)
    )


def prepare_source(path, source):
    df = load_table(path)

    missing = [col for col in REQUIRED_COLUMNS if col not in df.columns]
    if missing:
        raise ValueError(f"{source}: missing columns: {missing}")

    df = df[REQUIRED_COLUMNS].copy()

    for col in REQUIRED_COLUMNS:
        df[col] = clean_text(df[col])

    df["Antibiotic"] = (
        df["Antibiotic"]
        .str.lower()
        .str.replace(r"\s+", " ", regex=True)
    )

    raw_phenotype = df["Resistant Phenotype"].str.upper()
    mapped = raw_phenotype.map(PHENOTYPE_MAP)

    unknown = sorted(raw_phenotype[mapped.isna() & raw_phenotype.notna()].unique())
    if unknown:
        raise ValueError(
            f"{source}: unexpected phenotype labels found: {unknown}"
        )

    df["Resistant Phenotype"] = mapped
    df["source"] = source
    df["raw_row_id"] = source + ":" + (df.index + 2).astype(str)

    return df


In [ ]:
ncbi = prepare_source(ncbi_file, "NCBI")
bvbrc = prepare_source(bvbrc_file, "BV-BRC")

print("NCBI rows:", len(ncbi))
print("BV-BRC rows:", len(bvbrc))

biosample_overlap = set(ncbi["BioSample"].dropna()) & set(bvbrc["BioSample"].dropna())
assembly_overlap = set(ncbi["Assembly"].dropna()) & set(bvbrc["Assembly"].dropna())

print("BioSample overlap:", len(biosample_overlap))
print("Assembly overlap:", len(assembly_overlap))

merged = pd.concat([ncbi, bvbrc], ignore_index=True)

merged.to_csv(
    OUTPUT_DIR / "01_merged_with_provenance.csv",
    index=False,
)

print("Combined rows:", len(merged))
merged.head()


## 3. Check duplicates and phenotype conflicts

`Assembly + Antibiotic` is used as the isolate-antibiotic key.

- Repeated records with the same phenotype are treated as duplicates.
- Records with different phenotypes for the same `Assembly + Antibiotic` pair are saved separately for review.


In [ ]:
key = ["Assembly", "Antibiotic"]

valid_for_key = merged[
    merged["Assembly"].notna()
    & merged["Antibiotic"].notna()
    & merged["Resistant Phenotype"].notna()
].copy()

pair_stats = (
    valid_for_key
    .groupby(key, dropna=False)
    .agg(
        n_records=("Resistant Phenotype", "size"),
        n_labels=("Resistant Phenotype", "nunique"),
    )
    .reset_index()
)

conflict_keys = pair_stats.loc[
    pair_stats["n_labels"] > 1,
    key,
]

label_conflicts = valid_for_key.merge(
    conflict_keys,
    on=key,
    how="inner",
)

label_conflicts.to_csv(
    OUTPUT_DIR / "03_label_conflicts.csv",
    index=False,
)

# Remove conflicting pairs from the cleaned metadata.
conflict_index = pd.MultiIndex.from_frame(conflict_keys)
merged_index = pd.MultiIndex.from_frame(merged[key])

phenotype_clean = merged.loc[
    ~merged_index.isin(conflict_index)
].copy()

# Collapse repeated Assembly + Antibiotic records when they carry the same label.
phenotype_clean = (
    phenotype_clean
    .sort_values(["Assembly", "Antibiotic", "source"], na_position="last")
    .drop_duplicates(
        subset=["Assembly", "Antibiotic", "Resistant Phenotype"],
        keep="first",
    )
    .reset_index(drop=True)
)

phenotype_clean.to_csv(
    OUTPUT_DIR / "02_phenotype_clean.csv",
    index=False,
)

print("Duplicate isolate-antibiotic pairs:", int((pair_stats["n_records"] > 1).sum()))
print("Conflicting isolate-antibiotic pairs:", len(conflict_keys))
print("Clean rows:", len(phenotype_clean))


## 4. Antibiotic summary

Count the number of `S`, `I`, and `R` records for each antibiotic.


In [ ]:
antibiotic_summary = (
    phenotype_clean
    .groupby(["Antibiotic", "Resistant Phenotype"])
    .size()
    .unstack(fill_value=0)
    .reset_index()
)

for label in ["S", "I", "R"]:
    if label not in antibiotic_summary.columns:
        antibiotic_summary[label] = 0

antibiotic_summary = antibiotic_summary.rename(
    columns={
        "S": "n_S",
        "I": "n_I",
        "R": "n_R",
    }
)

antibiotic_summary["n_total"] = (
    antibiotic_summary["n_S"]
    + antibiotic_summary["n_I"]
    + antibiotic_summary["n_R"]
)

antibiotic_summary = antibiotic_summary[
    ["Antibiotic", "n_S", "n_I", "n_R", "n_total"]
].sort_values("n_total", ascending=False)

antibiotic_summary.to_csv(
    OUTPUT_DIR / "04_antibiotic_summary.csv",
    index=False,
)

antibiotic_summary


## Outputs

The notebook creates:

- `01_merged_with_provenance.csv` — combined NCBI and BV-BRC metadata with source information.
- `02_phenotype_clean.csv` — cleaned phenotype metadata used in the next steps.
- `03_label_conflicts.csv` — conflicting `Assembly + Antibiotic` records for review.
- `04_antibiotic_summary.csv` — phenotype counts for each antibiotic.


In [ ]:
# Download all outputs as a single ZIP file when running in Colab.
import shutil
from google.colab import files

archive = shutil.make_archive(
    "Salmonella_metadata_preprocessed",
    "zip",
    OUTPUT_DIR,
)

files.download(archive)
